# PyRestore 02 — Case A: restorative quality in Shenzhen (validated run)

Uses the matched Shenzhen dataset shipped in this repository (`data/shenzhen_prs11/`: 465
labelled captures from Ma & Kwan 2026, *Scientific Reports*; CV channel derived from the
reference segmentation, VLM channel scored against human PRS-11 ratings) and replays it through
`run_task` with precomputed analyzer tables, exercising the same validation and
spatial-screening layers notebook 01 demonstrates.

**Honesty notes:** the CV features are precomputed physical-composition estimates, not field
measurements; zero-shot VLM-vs-human agreement is uneven across dimensions (see the validation
report below); the map supports screening, not diagnosis.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML

from pyrestore import load_config, run_task

DATA = Path("data/shenzhen_prs11")

cfg = load_config({"vlm": {"cache_db": "cache/vlm_cache.sqlite", "offline": True}})
result = run_task(
    DATA / "manifest.csv",
    "tasks/prs11.yaml",
    cfg,
    cv_features=DATA / "cv_features.csv",      # precomputed physical measurements
    vlm_scores=DATA / "vlm_scores.csv",        # precomputed semantic interpretations
    reference=DATA / "reference.csv",          # human PRS-11 ratings
    check_files=False,                         # raw Shenzhen imagery is not redistributed
    out_dir="outputs/shenzhen_prs11",
)
print("validation_status:", result["validation_status"])
result["quality"]

validation_status: validated


,n_input_rows,n_valid_coordinates,n_cv_success,n_cv_error,n_vlm_success,n_vlm_error
0,465,465,465,0,465,0


## 1. Continuous validation against human PRS-11 ratings

In [2]:
validation = pd.read_csv(result["paths"]["validation"])
validation

,kind,target,reference_column,reference_type,pearson_r,pearson_p,pearson_ci_low,pearson_ci_high,spearman_rho,spearman_p,n,n_overlap,status
0,continuous,vlm_being_away,Being-away,human_prs11_ratings,0.5716,1.107358e-41,0.5070,0.6298,0.5802,3.518804e-43,465,465,validated
1,continuous,vlm_coherence,Coherence,human_prs11_ratings,0.1081,1.968359e-02,0.0174,0.1971,0.1254,6.759373e-03,465,465,validated
2,continuous,vlm_scope,Scope,human_prs11_ratings,0.3724,9.571423e-17,0.2914,0.4482,0.3603,1.071509e-15,465,465,validated
3,continuous,vlm_fascination,Fascination,human_prs11_ratings,0.1618,4.589849e-04,0.0720,0.2491,0.1794,1.005328e-04,465,465,validated
4,continuous,vlm_restorative_average,Average,human_prs11_ratings,0.3369,8.332166e-14,0.2538,0.4151,0.3407,4.196928e-14,465,465,validated


## 2. Spatial screening on the full labelled sample

In [3]:
stats = pd.read_csv(result["paths"]["spatial_stats"])
print(stats.to_string(index=False))
result["indicators"]["lisa_label"].value_counts()

              value_col  n_valid  k p_adjust  global_moran_i  global_moran_p    screening         candidate_col  n_candidate_low_clusters  n_candidate_high_clusters
vlm_restorative_average      465  8   fdr_bh        0.097858           0.001 lisa_low_low candidate_low_cluster                         0                          0


lisa_label
Not significant    465
Name: count, dtype: int64

In [4]:
HTML(filename=result["paths"]["map"])

## 3. Full re-extraction path (optional, needs imagery + API key)

```python
from pyrestore import run_task
result = run_task("data/shenzhen_prs11/manifest.csv", "tasks/prs11.yaml")   # manifest over local images
```

Raw Shenzhen imagery was obtained from the dataset authors and is not redistributed; build the
manifest with `pyrestore.manifest.manifest_from_directory` or an explicit CSV pointing at your
own local copies of the images.